# Window Functions, Spark SQL and Delta Lake

In this notebook we go one step further than the basics:

1. **Window functions**: calculations that look at the rows around each row (rankings, running totals, differences from the previous row).
2. **Spark SQL**: the same queries written in SQL instead of Python.
3. **Delta Lake**: saving the data as a real table, changing it, and going back in time.

We use the **Urban Essentials** data: 1,000 customers and their 2,000 orders. Run the cells from top to bottom (**Shift + Enter**).

> Parts 1 and 2 run anywhere. Part 3 (Delta Lake) needs **Databricks**.

## 0. Setup and data

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

def load_csv(path):
    """Read a CSV file of the course repository and return a Spark DataFrame."""
    pdf = pd.read_csv(f"{DATA_URL}/{path}")
    return spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None))

customers = load_csv("urban_essentials/customers.csv")
orders = load_csv("urban_essentials/orders.csv").withColumn("order_date", F.to_date("order_date"))

print("customers:", customers.count(), "| orders:", orders.count())

In [ ]:
display(orders.limit(5))

---
## Part 1: Window Functions

With `groupBy`, many rows become **one row per group**: for example, one total per customer.

A **window function** also works on groups of rows (the "window"), but it **keeps every row** and adds a new column. This is what you need for questions like:
- Is this the customer's first, second or third order?
- How much has the customer spent *up to* this order?
- How many days passed since the previous order?

A window is defined with `Window.partitionBy(...)` (which rows belong together) and `.orderBy(...)` (in which order).

### groupBy vs window: the difference
First with `groupBy`: one row per customer.

In [ ]:
orders.groupBy("customer_id").agg(F.round(F.sum("order_amount"), 2).alias("total_spent")).orderBy("customer_id").show(5)

Now with a window: every order is still there, and each row also shows the total of its customer.

In [ ]:
per_customer = Window.partitionBy("customer_id")

(orders
    .withColumn("total_spent", F.round(F.sum("order_amount").over(per_customer), 2))
    .orderBy("customer_id", "order_date")
    .show(8))

### Example 1: numbering the orders of each customer
`row_number()` gives 1 to the first row of each window, 2 to the second, and so on.

In [ ]:
by_date = Window.partitionBy("customer_id").orderBy("order_date")

numbered = orders.withColumn("order_number", F.row_number().over(by_date))
numbered.orderBy("customer_id", "order_number").show(8)

For example, keep only the **first order** of every customer:

In [ ]:
first_orders = numbered.filter(F.col("order_number") == 1)
print("customers with at least one order:", first_orders.count())

### Example 2: running total
How much has each customer spent **up to and including** each order? The window goes from the first row (`unboundedPreceding`) to the current row (`currentRow`).

In [ ]:
running = by_date.rowsBetween(Window.unboundedPreceding, Window.currentRow)

(orders
    .withColumn("running_total", F.round(F.sum("order_amount").over(running), 2))
    .orderBy("customer_id", "order_date")
    .show(8))

### Example 3: comparing with the previous row
`lag` reads a value from the **previous** row of the window. Here we compute how many days passed since the customer's previous order (the first order has no previous one, so the result is `null`).

In [ ]:
with_gap = (orders
    .withColumn("previous_order_date", F.lag("order_date").over(by_date))
    .withColumn("days_since_previous", F.datediff("order_date", "previous_order_date")))

with_gap.orderBy("customer_id", "order_date").show(8)

### Example 4: ranking inside a group
Which are the 3 best customers of each city? First we compute the total per customer (a `groupBy`), then we rank the customers inside each city (a window).

In [ ]:
spent = (orders
    .groupBy("customer_id").agg(F.round(F.sum("order_amount"), 2).alias("total_spent"))
    .join(customers, on="customer_id"))

by_city = Window.partitionBy("city").orderBy(F.desc("total_spent"))

top3 = (spent
    .withColumn("rank_in_city", F.rank().over(by_city))
    .filter(F.col("rank_in_city") <= 3)
    .select("city", "rank_in_city", "name", "total_spent")
    .orderBy("city", "rank_in_city"))

display(top3)

---
## Part 2: Spark SQL

Everything you do with DataFrames can also be written in **SQL**. Both are translated into the same plan and run on the same engine, so the speed is the same: you can choose what you find easier, and mix them.

To use a DataFrame in SQL, first give it a name with `createOrReplaceTempView`. The name only exists in this notebook session.

In [ ]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

### The same question, twice
The 5 cities with the highest total sales. First with DataFrames:

In [ ]:
(orders.join(customers, on="customer_id")
    .groupBy("city")
    .agg(F.round(F.sum("order_amount"), 2).alias("total_sales"))
    .orderBy(F.desc("total_sales"))
    .limit(5)
    .show())

Then with SQL:

In [ ]:
spark.sql("""
    SELECT c.city, ROUND(SUM(o.order_amount), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.city
    ORDER BY total_sales DESC
    LIMIT 5
""").show()

> On Databricks you can also write a cell in pure SQL, without `spark.sql`: start the cell with `%sql` on the first line. The result is shown as an interactive table.

### Window functions in SQL
The running total of Example 2, in SQL. The window is written with `OVER (PARTITION BY ... ORDER BY ...)`.

In [ ]:
spark.sql("""
    SELECT customer_id, order_date, order_amount,
           ROUND(SUM(order_amount) OVER (
               PARTITION BY customer_id
               ORDER BY order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ), 2) AS running_total
    FROM orders
    ORDER BY customer_id, order_date
""").show(8)

---
## Part 3: Delta Lake (Databricks only)

Until now our data lived only in memory, for the duration of the notebook. Now we **save** it as a table. On Databricks every table is a **Delta Lake** table: Parquet files plus a **transaction log** that records every change. Thanks to the log we can update and delete rows like in a database, see the history of the table, and go back to an old version.

### Save the orders as a table
We first remove the table if it exists, so that the notebook gives the same results every time you run it.

In [ ]:
spark.sql("DROP TABLE IF EXISTS urban_orders")
orders.write.saveAsTable("urban_orders")

display(spark.sql("SELECT COUNT(*) AS orders FROM urban_orders"))

The table is now permanent: you can find it in **Catalog > workspace > default > urban_orders**, and it is still there tomorrow. `DESCRIBE DETAIL` shows that its format is `delta`, and how many files it is made of.

In [ ]:
display(spark.sql("DESCRIBE DETAIL urban_orders").select("format", "numFiles", "sizeInBytes"))

### Change the table: UPDATE and DELETE
Two changes, like in a real company:
1. Order 1 was typed wrong: the amount is **35.11**, not 351.06.
2. Orders under 20 euros were test orders and must be removed.

In [ ]:
spark.sql("UPDATE urban_orders SET order_amount = 35.11 WHERE order_id = 1")
spark.sql("DELETE FROM urban_orders WHERE order_amount < 20")

display(spark.sql("SELECT COUNT(*) AS orders, MIN(order_amount) AS min_amount FROM urban_orders"))

### The history of the table
Every change is a new **version** of the table, recorded in the transaction log.

In [ ]:
display(spark.sql("DESCRIBE HISTORY urban_orders").select("version", "timestamp", "operation"))

### Time travel
We can read the table **as it was** at an older version. Version 0 is the table just after we created it.

In [ ]:
display(spark.sql("""
    SELECT 'now' AS version, COUNT(*) AS orders, MAX(CASE WHEN order_id = 1 THEN order_amount END) AS order_1_amount
    FROM urban_orders
    UNION ALL
    SELECT 'version 0', COUNT(*), MAX(CASE WHEN order_id = 1 THEN order_amount END)
    FROM urban_orders VERSION AS OF 0
"""))

### Undo: go back to an old version
If a change was a mistake, `RESTORE` brings the whole table back to an old version. The restore is itself a new version in the history: nothing is lost.

In [ ]:
spark.sql("RESTORE TABLE urban_orders TO VERSION AS OF 0")

display(spark.sql("SELECT COUNT(*) AS orders FROM urban_orders"))
display(spark.sql("DESCRIBE HISTORY urban_orders").select("version", "operation"))

---
## Summary

| What | DataFrame (Python) | SQL |
|------|--------------------|-----|
| number the rows of each group | `F.row_number().over(w)` | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` |
| running total | `F.sum(...).over(w.rowsBetween(...))` | `SUM(...) OVER (... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| previous row | `F.lag("col").over(w)` | `LAG(col) OVER (...)` |
| use a DataFrame in SQL | `df.createOrReplaceTempView("name")` | |
| save as a Delta table | `df.write.saveAsTable("name")` | `CREATE TABLE ... AS SELECT ...` |
| change rows | | `UPDATE`, `DELETE` |
| history and time travel | | `DESCRIBE HISTORY`, `VERSION AS OF`, `RESTORE` |

## Exercises

1. For each customer, find their **largest** order (hint: a window ordered by `order_amount` descending, then `row_number() == 1`).
2. Compute, for each order, the difference between its amount and the **average** order amount of the same customer.
3. Write exercise 1 in **SQL**.
4. In the `urban_orders` table, double the amount of all orders of 2024 with an `UPDATE`. Check the history, then use time travel to compare the total sales before and after.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4